# CALSNIC cortex — test-split meshes

Ground truth, the PCA-172 oracle, and the three final Fourier experiments, on the
**locked test split** at extraction resolution 256.

| panel | run | readout |
| --- | --- | --- |
| C1 Fourier compact z256 | `calsnic_compact_fourier_z256_exact` | `global_only` |
| C2 Fourier compact z512 | `calsnic_compact_fourier_z512_exact` | `global_only` |
| C3 Fourier grid-free | `calsnic_fourier_global_z256_exact` | `single` |

`global_only` is the winning readout for the two-branch runs: it beat
`fused_smooth_gate` on every geometry metric at every checkpoint, in both runs.


In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import trimesh

MULTIRES_ROOT = Path('/mnt/bulk10tb/Deep3DComp/CALSNIC/control_L_exact_multires_v1')
HASHGRID_ROOT = Path('/mnt/bulk10tb/Deep3DComp/CALSNIC/control_L_exact_instant_ngp_v1')
MANIFEST_PATH = MULTIRES_ROOT / 'manifests/calsnic_control_L_exact.csv'

# run directory name -> (panel label, readout variant)
EXPERIMENTS = {
    'calsnic_compact_fourier_z256_exact': ('C1 Fourier compact z256', 'global_only'),
    'calsnic_compact_fourier_z512_exact': ('C2 Fourier compact z512', 'global_only'),
    'calsnic_fourier_global_z256_exact':  ('C3 Fourier grid-free',    'single'),
}
EVAL_DIR = 'final_test/{run}_e2200_r256'
PCA_LABEL = 'PCA-172 oracle'
# Rank test scans by this panel's ASSD to pick the example; None -> first scan.
REFERENCE_LABEL = 'C2 Fourier compact z512'
EXAMPLE_MODE = 'median'      # best | median | worst
TEST_SCAN_ID = None          # set a scan id string to pin the example

# Display only, never affects a metric. None renders every triangle.
# Do NOT subsample faces: dropping every Nth triangle discards its neighbours
# and the surface renders as loose specks rather than a mesh.
PLOT_FACE_LIMIT = None

manifest = pd.read_csv(MANIFEST_PATH).set_index('scan_id')
print(f'manifest: {len(manifest)} scans, {int((manifest["split"] == "test").sum())} in test')

In [ ]:
def eval_dir(run):
    return HASHGRID_ROOT / EVAL_DIR.format(run=run)

def mesh_path(run, variant, scan_id):
    return eval_dir(run) / 'meshes' / variant / 'test' / f'{scan_id}.ply'

def load_mesh(path):
    mesh = trimesh.load(Path(path), process=False)
    if isinstance(mesh, trimesh.Scene):
        mesh = trimesh.util.concatenate(tuple(mesh.geometry.values()))
    if not isinstance(mesh, trimesh.Trimesh) or not len(mesh.faces):
        raise ValueError(f'Invalid mesh: {path}')
    return mesh

# One PCA directory; the oracle is identical in every evaluation that wrote it.
PCA_DIR = next(
    (eval_dir(run) / 'meshes' / 'pca' / 'test'
     for run in EXPERIMENTS if (eval_dir(run) / 'meshes' / 'pca' / 'test').is_dir()),
    MULTIRES_ROOT / 'final_test/multires128_best_mesh_r256/meshes/pca/test',
)

def available_scans():
    """Scan ids present for the oracle and for every experiment."""
    sets = [{p.stem for p in PCA_DIR.glob('*.ply')}]
    for run, (_label, variant) in EXPERIMENTS.items():
        directory = eval_dir(run) / 'meshes' / variant / 'test'
        sets.append({p.stem for p in directory.glob('*.ply')} if directory.is_dir() else set())
    return sorted(set.intersection(*sets)) if sets else []

def test_metrics():
    frames = []
    for run, (label, variant) in EXPERIMENTS.items():
        path = eval_dir(run) / 'per_scan_metrics.csv'
        if not path.is_file():
            continue
        frame = pd.read_csv(path).query('split == "test"')
        frame = frame[frame['variant'].isin([variant, 'pca_oracle'])].copy()
        frame['panel'] = [PCA_LABEL if v == 'pca_oracle' else label for v in frame['variant']]
        frames.append(frame)
    if not frames:
        return pd.DataFrame()
    metrics = pd.concat(frames, ignore_index=True)
    # The oracle repeats once per evaluation directory; keep a single copy.
    is_oracle = metrics['panel'].eq(PCA_LABEL)
    return pd.concat([metrics[~is_oracle],
                      metrics[is_oracle].drop_duplicates(subset=['scan_id'])],
                     ignore_index=True)

SCANS = available_scans()
METRICS = test_metrics()
missing = [label for run, (label, variant) in EXPERIMENTS.items()
           if not (eval_dir(run) / 'meshes' / variant / 'test').is_dir()]
print(f'test scans shared by all panels: {len(SCANS)}')
if missing:
    print('still evaluating (re-run this cell when they finish):', ', '.join(missing))

In [ ]:
COLUMNS = ['assd_mm', 'hd95_mm', 'fscore_1mm', 'normal_absolute_cosine',
           'high_curvature_gt_to_prediction_mm', 'volume_relative_error',
           'predicted_connected_components']

def choose_scan():
    if TEST_SCAN_ID is not None:
        if TEST_SCAN_ID not in SCANS:
            raise KeyError(f'{TEST_SCAN_ID} is not available for every panel')
        return TEST_SCAN_ID
    if not SCANS:
        return None
    frame = METRICS.query('panel == @REFERENCE_LABEL and scan_id in @SCANS')
    if frame.empty:
        return SCANS[0]
    frame = frame.sort_values('assd_mm')
    position = {'best': 0, 'median': len(frame) // 2, 'worst': len(frame) - 1}[EXAMPLE_MODE]
    return str(frame.iloc[position]['scan_id'])

test_scan = choose_scan()
print('test example:', test_scan)
if not METRICS.empty:
    order = [PCA_LABEL] + [label for label, _ in EXPERIMENTS.values()]
    table = (METRICS.query('scan_id in @SCANS').groupby('panel')[COLUMNS].mean()
                    .reindex([p for p in order if p in set(METRICS['panel'])]).round(4))
    table['n'] = METRICS.query('scan_id in @SCANS').groupby('panel').size()
    display(table)

In [ ]:
PALETTE = ['#999999', '#4477AA', '#EE7733', '#228833', '#CC6677']

def component_count(mesh):
    labels = trimesh.graph.connected_component_labels(
        mesh.face_adjacency, node_count=len(mesh.faces))
    return int(labels.max() + 1) if len(labels) else 0

def cluster_decimate(mesh, cell_mm):
    """Merge vertices sharing a `cell_mm` grid cell; keeps the surface connected."""
    vertices, faces = np.asarray(mesh.vertices), np.asarray(mesh.faces)
    key = np.floor((vertices - vertices.min(axis=0)) / cell_mm).astype(np.int64)
    _unique, inverse = np.unique(key, axis=0, return_inverse=True)
    merged = np.zeros((inverse.max() + 1, 3))
    counts = np.bincount(inverse, minlength=inverse.max() + 1)
    np.add.at(merged, inverse, vertices)
    merged /= counts[:, None]
    remapped = inverse[faces]
    keep = ((remapped[:, 0] != remapped[:, 1]) & (remapped[:, 1] != remapped[:, 2])
            & (remapped[:, 0] != remapped[:, 2]))
    remapped = np.unique(np.sort(remapped[keep], axis=1), axis=0)
    return trimesh.Trimesh(vertices=merged, faces=remapped, process=False)

def decimate_for_plot(mesh):
    """Display-only reduction. Always returns a surface, never loose triangles."""
    if not PLOT_FACE_LIMIT or len(mesh.faces) <= PLOT_FACE_LIMIT:
        return mesh
    # Face count falls monotonically as the cell grows, so bisection is valid.
    edge = float(np.median(mesh.edges_unique_length))
    low, high, best = edge * 0.5, edge * 32.0, None
    for _ in range(14):
        cell = 0.5 * (low + high)
        candidate = cluster_decimate(mesh, cell)
        if len(candidate.faces) <= PLOT_FACE_LIMIT:
            best, high = candidate, cell
        else:
            low = cell
        if high - low < edge * 0.05:
            break
    return best if best is not None else cluster_decimate(mesh, high)

def load_panels(scan_id):
    panels = {'Target (ground truth)': load_mesh(manifest.loc[scan_id, 'mesh_path_mm'])}
    pca = PCA_DIR / f'{scan_id}.ply'
    if pca.is_file():
        panels[PCA_LABEL] = load_mesh(pca)
    for run, (label, variant) in EXPERIMENTS.items():
        path = mesh_path(run, variant, scan_id)
        if path.is_file():
            panels[label] = load_mesh(path)
    return panels

def show(scan_id):
    if scan_id is None:
        print('No test meshes available yet.')
        return
    panels = load_panels(scan_id)
    columns = len(panels)
    fig = make_subplots(
        rows=1, cols=columns,
        specs=[[{'type': 'scene'}] * columns],
        subplot_titles=[
            f'{name}<br><sub>V={len(m.vertices):,} F={len(m.faces):,} '
            f'CC={component_count(m)} area={m.area:,.0f} mm²</sub>'
            for name, m in panels.items()],
        horizontal_spacing=0.005,
    )
    for index, (name, mesh) in enumerate(panels.items()):
        mesh = decimate_for_plot(mesh)
        vertices, faces = np.asarray(mesh.vertices), np.asarray(mesh.faces)
        fig.add_trace(go.Mesh3d(
            x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
            i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
            name=name, color=PALETTE[index % len(PALETTE)], flatshading=False,
            lighting=dict(ambient=0.45, diffuse=0.8, specular=0.15, roughness=0.8),
        ), row=1, col=index + 1)
        scene = 'scene' if index == 0 else f'scene{index + 1}'
        fig.update_layout(**{scene: dict(
            aspectmode='data', xaxis_visible=False, yaxis_visible=False,
            zaxis_visible=False, camera=dict(eye=dict(x=1.4, y=1.4, z=0.9)))})
    fig.update_layout(height=560, width=380 * columns, showlegend=False,
                      title=f'TEST split — {scan_id}')
    fig.show()

show(test_scan)